# Cattle Re-ID — Test de localización del hocico (Grad-CAM)

Objetivo: ver **con el ojo** si el Grad-CAM del **VGG16_BN** (entrenado en hocicos) cae sobre el **hocico** cuando le pasás **caras enteras** de Ahmed. Es un test **visual**, no una métrica.

Para cada cara genera tres paneles lado a lado:
**cara original | overlay del Grad-CAM | preview del blur** (hocico nítido / resto borroso).

**Cómo leerlo:**
- Si el calor (rojo/amarillo) cae sobre el **hocico** en la mayoría de las ~20 caras → el localizador sirve, seguimos al pipeline de blur sobre todo Ahmed.
- Si cae en ojos, orejas, fondo o disperso → el VGG no generaliza de primeros planos a caras enteras; hay que repensar el localizador antes de construir nada.

**Antes de correr:**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4 alcanza).
2. El **VGG sano** (`vgg16_bn_wce_aug_s0_best.pt`) tiene que estar en tu Drive. Si corriste la replicación con `colab_runner.ipynb`, ya quedó en `MyDrive/cattle_reid/outputs/checkpoints/` (la celda 2 lo busca solo).
3. El zip de caras en `MyDrive/cattle_reid/dataset_caras_bovinos.zip`, organizado por individuo.

Autocontenido: no hace falta clonar el repo para esta prueba.

## 0. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}. Ajustá DRIVE_ROOT.'
print('DRIVE_ROOT:', DRIVE_ROOT)

## 2. Configuración — rutas y parámetros

Ajustá acá si tus archivos están en otro lado. Los parámetros que más vas a tocar son `CAM_THRESHOLD` (qué tan grande sale la máscara del hocico) y `N_IMAGES`.

In [ ]:
# --- rutas ---
CAND_VGG = [
    DRIVE_ROOT / 'outputs' / 'checkpoints' / 'vgg16_bn_wce_aug_s0_best.pt',  # si corriste replicación en Colab
    DRIVE_ROOT / 'vgg16_bn_wce_aug_s0_best.pt',                              # o subido suelto a Drive
]
VGG_CKPT = next((p for p in CAND_VGG if p.is_file()), None)
assert VGG_CKPT, f'No encuentro el VGG. Subí vgg16_bn_wce_aug_s0_best.pt a Drive. Busqué en: {CAND_VGG}'

FACES_ZIP = DRIVE_ROOT / 'dataset_caras_bovinos.zip'
assert FACES_ZIP.is_file(), f'No encuentro {FACES_ZIP}.'

WORK_DIR  = Path('/content/gradcam_test')            # disco local (rápido, efímero)
OUT_DIR   = DRIVE_ROOT / 'gradcam_test'              # la grilla se guarda acá (persiste)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- parámetros del test ---
N_IMAGES          = 20      # cuántas caras muestrear
IMAGE_SIZE        = 224     # igual que la Etapa 2
USE_IMAGENET_NORM = True    # igual que la Etapa 2
NUM_CLASSES_CKPT  = 268     # el VGG tiene cabeza de Zenodo (268); para Grad-CAM da igual
CAM_THRESHOLD     = 0.5     # umbral del heatmap [0,1] para la máscara del hocico
SEED              = 0

print('VGG  :', VGG_CKPT)
print('CARAS:', FACES_ZIP)

## 3. Elegir y extraer SOLO las caras del test (no descomprime todo)

El zip puede tener miles de imágenes; para mirar ~20 no hace falta extraerlo entero.
Esta celda lee el **índice** del zip (instantáneo, sin descomprimir), muestrea
`N_IMAGES` rutas y extrae **solo esas** a disco local.

In [ ]:
import zipfile
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

faces_root = WORK_DIR / 'faces_subset'
faces_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(FACES_ZIP) as z:
    # índice completo SIN descomprimir
    members = [m for m in z.namelist()
               if not m.endswith('/') and Path(m).suffix.lower() in IMG_EXTS]
    n_total = len(members)
    n_ind = len({Path(m).parent.as_posix() for m in members})  # subcarpetas = individuos
    print(f'en el zip: {n_total} imágenes | {n_ind} individuos (NO se descomprimió nada)')

    # muestrear N_IMAGES rutas (1 por individuo distinto cuando se pueda)
    import numpy as np
    rng = np.random.default_rng(SEED)
    by_ind = {}
    for m in members:
        by_ind.setdefault(Path(m).parent.as_posix(), []).append(m)
    inds = list(by_ind)
    rng.shuffle(inds)
    picked = []
    for ind in inds:                      # primero una de cada individuo
        picked.append(rng.choice(by_ind[ind]))
        if len(picked) >= N_IMAGES:
            break
    picked = picked[:N_IMAGES]

    # extraer SOLO esas
    sample_paths = []
    for m in picked:
        z.extract(m, faces_root)
        sample_paths.append(faces_root / m)

print(f'extraídas {len(sample_paths)} imágenes a {faces_root}')

## 4. Modelo VGG16_BN + Grad-CAM (definiciones)

Gancho en `features[40]` (última conv de VGG16_BN). Clase para el gradiente = **argmax del propio VGG** (las vacas de Ahmed no están entre las 268 que vio; querés ver dónde mira para decidir lo que sea).

In [ ]:
import numpy as np
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torchvision import models as tvm
from torchvision import transforms
import matplotlib
try:
    _JET = matplotlib.colormaps['jet']
except Exception:
    import matplotlib.cm as _cm; _JET = _cm.get_cmap('jet')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def load_vgg(ckpt_path, num_classes, device):
    model = tvm.vgg16_bn(weights=None)
    in_f = model.classifier[6].in_features
    model.classifier[6] = torch.nn.Linear(in_f, num_classes)
    obj = torch.load(ckpt_path, map_location='cpu')
    state = obj['model_state'] if isinstance(obj, dict) and 'model_state' in obj else obj
    miss, unexp = model.load_state_dict(state, strict=False)
    if miss or unexp:
        print(f'[load] missing={len(miss)} unexpected={len(unexp)} (ok si solo difiere la cabeza)')
    return model.eval().to(device)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.acts = None; self.grads = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.acts = o.detach()
    def _bwd(self, m, gi, go): self.grads = go[0].detach()
    def __call__(self, x, class_idx=None):
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(logits.argmax(1).item())
        self.model.zero_grad()
        logits[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        return cam, class_idx


def build_tf():
    ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()]
    if USE_IMAGENET_NORM:
        ops.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return transforms.Compose(ops)


def heatmap_overlay(pil, cam):
    base = np.asarray(pil.resize((IMAGE_SIZE, IMAGE_SIZE)).convert('RGB')) / 255.0
    jet = _JET(cam)[..., :3]
    return np.clip(0.55 * base + 0.45 * jet, 0, 1)


def masked_preview(pil, cam, thr):
    base = pil.resize((IMAGE_SIZE, IMAGE_SIZE)).convert('RGB')
    blurred = base.filter(ImageFilter.GaussianBlur(radius=8))
    mask = (cam >= thr).astype(np.float32)[..., None]
    return np.clip(np.asarray(base)/255.0 * mask + np.asarray(blurred)/255.0 * (1 - mask), 0, 1)


def sample_faces(root, n, seed):
    allp = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS]
    rng = np.random.default_rng(seed)
    if len(allp) <= n:
        return sorted(allp)
    return [allp[i] for i in sorted(rng.choice(len(allp), size=n, replace=False))]

print('definiciones cargadas | device =', DEVICE)

## 5. Cargar el VGG y construir el Grad-CAM

In [ ]:
model = load_vgg(VGG_CKPT, NUM_CLASSES_CKPT, DEVICE)
cam_engine = GradCAM(model, model.features[40])   # última conv de VGG16_BN
tf = build_tf()
print('VGG cargado y Grad-CAM enganchado en features[40].')

## 6. Correr el test (overlays + preview de blur)

Genera la grilla de `N_IMAGES` caras y la guarda en Drive (`gradcam_test/`). **Mirá la columna del medio**: ¿el calor cae sobre el hocico?

In [ ]:
import matplotlib.pyplot as plt

def run_test(n, threshold, faces_list=None, save_name=None):
    faces = (sample_paths if faces_list is None else faces_list)[:n]
    print(f'procesando {len(faces)} caras (thr={threshold})...')
    rows, cols = len(faces), 3
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    if rows == 1: axes = axes[None, :]
    for r, fp in enumerate(faces):
        pil = Image.open(fp).convert('RGB')
        x = tf(pil).unsqueeze(0).to(DEVICE)
        cam, cls = cam_engine(x, class_idx=None)
        panels = [
            (np.asarray(pil.resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0, f'{fp.parent.name}/{fp.name}'),
            (heatmap_overlay(pil, cam), f'Grad-CAM (cls={cls})'),
            (masked_preview(pil, cam, threshold), f'preview blur (thr={threshold})'),
        ]
        for c, (img, title) in enumerate(panels):
            axes[r, c].imshow(img); axes[r, c].set_title(title, fontsize=8); axes[r, c].axis('off')
    fig.tight_layout()
    name = save_name or f'gradcam_grid_thr{threshold}.png'
    out = OUT_DIR / name
    fig.savefig(out, dpi=110, bbox_inches='tight')
    print('grilla guardada en', out)
    plt.show()

run_test(N_IMAGES, CAM_THRESHOLD)

## 7. Ajustar y volver a correr (opcional)

Si la máscara del hocico (3ra columna) sale muy chica, bajá el umbral; si agarra de más, subilo. No reentrena nada, solo recalcula la máscara.

In [ ]:
# probá otro umbral sin tocar lo demás:
run_test(N_IMAGES, threshold=0.3)
# run_test(N_IMAGES, threshold=0.7)